# Gulfstream walkthrough — equities (`equity_eod`)

Same public-API Graph **1** → Graph **2** story as the YCS notebook, on a
**stock-level log-return + rolling-vol** panel (all EqIndex universes; no raw
price levels), plus Parts F–L for search / tests / window / classical / TFT /
curve dimred, **Part M** for product artifacts, and **Part N** for Graph 2
retrain score methods.

| Part | Focus |
|------|--------|
| A | PCA → Graph 1 + Graph 2 |
| B | Kernel PCA → Graph 1 + Graph 2 |
| C | DMD → Graph 1 + Graph 2 |
| D | t-SNE → Graph 1 + Graph 2 |
| E | UMAP → Graph 1 + Graph 2 (optional) |
| F | Binseg / BottomUp / WBS / BOCPD search |
| G | Statistical tests |
| H | ESS window |
| I | Classical hard-label detectors + Graph 2 |
| J | Classical models as soft dimred |
| K | TFT attention embeddings as dimred (+ optional Graph 2) |
| L | Curve / ICA dimred |
| M | Product: uncertainty + CI ribbons, Excel, events, streaming, panel |
| N | Graph 2 retrain scores (`mse_on_diff`, `energy_split`, `mmd_split`, …) |
| — | Comparison (covering + ARI + F1 vs PCA) |

**Database:** `D:/data/duckdb/equity_eod_data.duckdb` · **table:** `equity_eod`

> Close DBeaver if the file is locked; falls back to `equity_eod_data_copy.duckdb`.


## 0. Project setup


In [ ]:
from __future__ import annotations

from pathlib import Path
import sys

import duckdb
import pandas as pd
import polars as pl
from plotnine import aes, geom_line, ggplot, labs, theme_bw, facet_wrap, theme

NOTEBOOK_DIR = Path.cwd()
if (NOTEBOOK_DIR / "pyproject.toml").exists():
    ROOT = NOTEBOOK_DIR
elif (NOTEBOOK_DIR.parent / "pyproject.toml").exists():
    ROOT = NOTEBOOK_DIR.parent
else:
    ROOT = Path(r"D:/Code/gulfstream")

SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

EQ_CANDIDATES = [
    Path(r"D:/data/duckdb/equity_eod_data.duckdb"),
    Path(r"D:/data/duckdb/equity_eod_data_copy.duckdb"),
]
EQ_DB = next((p for p in EQ_CANDIDATES if p.exists()), EQ_CANDIDATES[0])
OUT_DIR = ROOT / "outputs" / "notebooks" / "equity"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("ROOT =", ROOT)
print("EQ_DB =", EQ_DB, "exists =", EQ_DB.exists())


## 1. Schema & coverage


In [ ]:
def open_equity() -> duckdb.DuckDBPyConnection:
    try:
        return duckdb.connect(str(EQ_DB), read_only=True)
    except Exception as exc:
        raise RuntimeError(
            f"Could not open {EQ_DB}. Close DBeaver / other DuckDB clients and retry.\n{exc}"
        ) from exc

con = open_equity()
print(con.execute("DESCRIBE equity_eod").pl())
print(
    con.execute(
        """
        SELECT EqIndex, COUNT(*) AS n, COUNT(DISTINCT Stock) AS stocks,
               MIN(Index) AS dmin, MAX(Index) AS dmax
        FROM equity_eod
        GROUP BY 1
        ORDER BY n DESC
        """
    ).pl()
)
con.close()


In [ ]:
START, END = "2007-01-01", "2012-12-31"

con = open_equity()
long_px = con.execute(
    f"""
    SELECT CAST(Index AS DATE) AS date,
           EqIndex AS eq_index,
           Stock AS ticker,
           Close AS close
    FROM equity_eod
    WHERE Index >= '{START}'
      AND Index <= '{END}'
    ORDER BY date, eq_index, ticker
    """
).pl()
con.close()

# Daily log returns per stock (log-diff of close within each index/ticker series)
long_px = long_px.sort(["eq_index", "ticker", "date"])
long_ret = long_px.with_columns(
    pl.col("close").log().diff().over(["eq_index", "ticker"]).alias("log_ret")
).drop_nulls(subset=["log_ret"])
long_ret = long_ret.with_columns(
    (pl.col("eq_index") + ":" + pl.col("ticker")).alias("stock_id")
)
long_ret = long_ret.select(["date", "eq_index", "ticker", "stock_id", "log_ret"])

eq_indices = sorted(long_ret["eq_index"].unique().to_list())
print(f"date window: {START} → {END}")
print(f"indices: {eq_indices}")
print(f"long_ret rows: {long_ret.height:,}")
long_ret.head(3)

## 2. Build a wide stock return panel

**All EqIndex universes** over the GFC window — one column per stock (`EqIndex:ticker`).
Pick **`N_TICKERS_PER_INDEX`** names with the **longest return history** per index (not
alphabetical) so the complete-case panel starts as early as possible.
Only **log returns** enter the wide frame (no price levels).


In [ ]:
from gulfstream.common import frames

N_TICKERS_PER_INDEX = 7  # 7 stocks × 6 indices → 42 names (min 5 per index)

selected = (
    long_ret.group_by(["eq_index", "ticker", "stock_id"])
    .agg(pl.len().alias("n_obs"))
    .sort(["eq_index", "n_obs"], descending=[False, True])
    .group_by("eq_index")
    .head(N_TICKERS_PER_INDEX)
)
stock_ids = selected["stock_id"].to_list()

wide = (
    long_ret.filter(pl.col("stock_id").is_in(stock_ids))
    .select(["date", "stock_id", "log_ret"])
    .pivot(values="log_ret", index="date", on="stock_id", aggregate_function="first")
    .sort("date")
)
wide = frames.ensure_date_column(wide)
tickers = [c for c in frames.feature_columns(wide) if c in stock_ids]
wide = wide.select(["date", *tickers]).drop_nulls()
print("wide shape:", wide.shape)
print(f"stocks: {len(tickers)} ({N_TICKERS_PER_INDEX} per index, coverage-ranked)")
print(f"date range: {wide['date'].min()} → {wide['date'].max()}")
print("sample:", tickers[:4])
wide.head(3)

## 3. Visualize stock log returns


In [ ]:
plot_tickers = tickers[:6]
long_plot = (
    wide.unpivot(index="date", on=plot_tickers, variable_name="stock", value_name="log_ret")
    .to_pandas()
)
long_plot["date"] = pd.to_datetime(long_plot["date"])

(
    ggplot(long_plot, aes("date", "log_ret", color="stock"))
    + geom_line(size=0.35)
    + theme_bw()
    + theme(figure_size=(11, 4))
    + labs(title=f"Stock log returns ({START} → {END})", x="", y="log return")
)

## 4. Feature engineering

Per-stock **log returns** plus 20-day rolling **vol** → full feature matrix for
gulfstream (stationary inputs; no price levels).


In [ ]:
vol_window = 20
vol_cols: list[str] = []
features_df = wide.select("date")
for c in tickers:
    v = f"{c}_vol"
    vol_cols.append(v)
    features_df = features_df.with_columns(
        wide[c].alias(c),
        wide[c].rolling_std(vol_window).alias(v),
    )

features_df = features_df.drop_nulls()
print("features:", features_df.shape, "n_feat:", frames.n_features(features_df))
print(f"  returns: {len(tickers)}  vol: {len(vol_cols)}")
print(f"date range: {features_df['date'].min()} → {features_df['date'].max()}")
plot_cols = tickers[:3]
features_df.head(3)


## 5. Feature snapshot


In [ ]:
viz_cols = tickers[:2] + vol_cols[:2]
viz_cols = [c for c in viz_cols if c in features_df.columns]
long_f = (
    features_df.select(["date", *viz_cols])
    .unpivot(index="date", on=viz_cols, variable_name="series", value_name="value")
    .to_pandas()
)
long_f["date"] = pd.to_datetime(long_f["date"])

(
    ggplot(long_f, aes("date", "value", color="series"))
    + geom_line(size=0.35)
    + facet_wrap("~series", scales="free_y", ncol=1)
    + theme_bw()
    + theme(figure_size=(10, 2.0 * len(viz_cols)), legend_position="none")
    + labs(title="Log returns + rolling vol fed to gulfstream", x="", y="")
)


## 6. Shared helpers (public API)

Equity defaults use a slightly looser MMD gate so breaks survive in this window.


In [ ]:
import copy
from IPython.display import Image, display

from gulfstream import (
    plot_regimes,
    refine_regimes,
    regime_intervals,
    run_single_segmentation,
    seed_regimes_from_results,
)
from gulfstream.common import frames, utils
from gulfstream.common.options import (
    ClassicalDetector,
    DetectionBackend,
    SearchMethod,
    StatTest,
)
from gulfstream.metrics.evaluation import (
    adjusted_rand_index,
    breakpoint_precision_recall_f1,
    covering_metric,
)


def load_core_params(img_dir: Path) -> dict:
    """Validated Graph 1 core params with notebook-friendly metrics."""
    params = utils.read_config_yaml(
        str(ROOT / "config" / "graph1" / "default_core.yaml"),
        img_dir=str(img_dir),
        log_dir=str(ROOT / "outputs" / "logs"),
    )
    params["test_num"] = 0
    params["metrics"]["mode"] = "display_and_write"
    params["metrics"]["plot"] = True
    params["metrics"]["dir"] = str(img_dir)
    params["metrics"]["image_dir"] = str(img_dir)
    params["robustness"]["enabled"] = False
    params["stability"]["enabled"] = False
    return params


def with_dimred(params: dict, method: str) -> dict:
    out = copy.deepcopy(params)
    method = method.lower()
    out["algo"]["dimred"] = [method]
    if method == "kpca":
        out["algo"]["kpca_kernel_params"] = [{"kernel": "rbf", "gamma": "median"}]
    elif method == "dmd":
        out["algo"]["dmd_stride"] = [5]
        out["algo"]["dmd_rolling_window"] = [20]
    elif method == "ica":
        out["algo"]["rank"] = [3]
        out["algo"]["rank_selection_method"] = ["user_specified"]
        out["algo"]["random_state"] = [42]
        out["algo"]["ica_max_iter"] = [200]
    elif method == "fpca":
        out["algo"]["rank_selection_method"] = ["explained_variance"]
        out["algo"]["threshold"] = [0.9]
        out["algo"]["fpca_smooth_window"] = [3]
    elif method == "nelson_siegel":
        out["algo"]["ns_lambda"] = [0.0609]
    elif method == "dynamic_factor":
        out["algo"]["rank"] = [2]
        out["algo"]["rank_selection_method"] = ["user_specified"]
        out["algo"]["factor_order"] = [1]
        out["algo"]["df_maxiter"] = [30]
    elif method == "tsne":
        out["algo"]["rank"] = [2]
        out["algo"]["rank_selection_method"] = ["user_specified"]
        out["algo"]["tsne_perplexity"] = [30.0]
        out["algo"]["tsne_n_iter"] = [250]
        out["algo"]["random_state"] = [42]
    elif method == "umap":
        out["algo"]["rank"] = [2]
        out["algo"]["rank_selection_method"] = ["user_specified"]
        out["algo"]["umap_num_neighbors"] = [15]
        out["algo"]["umap_min_dist"] = [0.1]
        out["algo"]["umap_metric"] = ["euclidean"]
        out["algo"]["random_state"] = [42]
    elif method != "pca":
        raise ValueError(f"Unsupported dimred: {method}")
    return out


def with_search(params: dict, method, **algo_extras) -> dict:
    out = copy.deepcopy(params)
    out["algo"]["search_method"] = [str(method)]
    for k, v in algo_extras.items():
        out["algo"][k] = v if isinstance(v, list) else [v]
    return out


def with_test(params: dict, choice) -> dict:
    out = copy.deepcopy(params)
    out["test"]["choice"] = [str(choice)]
    return out


def with_ess_window(
    params: dict,
    *,
    ess_fraction: float = 0.25,
    min_window: int = 20,
    max_window: int = 100,
) -> dict:
    out = copy.deepcopy(params)
    out["test"]["window"] = [
        {
            "method": "ess",
            "ess_fraction": ess_fraction,
            "min_window": min_window,
            "max_window": max_window,
        }
    ]
    return out


def with_classical(
    params: dict,
    detector,
    *,
    regimes: int | None = 3,
    min_regime_length: int = 20,
    **algo_extras,
) -> dict:
    """Hard-label classical backend (former --mode classical)."""
    out = copy.deepcopy(params)
    out["algo"]["detection_backend"] = [str(DetectionBackend.CLASSICAL)]
    out["algo"]["regime_detection_algorithm"] = [str(detector)]
    out["algo"]["dimred"] = ["raw"]
    out["algo"]["feature_map_approx_method"] = ["raw"]
    out["algo"]["post_processing_method"] = ["majority_voting"]
    out["algo"]["min_regime_length"] = [min_regime_length]
    out["algo"]["include_last_regime"] = [True]
    if regimes is not None:
        out["algo"]["regimes"] = [regimes]
    for k, v in algo_extras.items():
        out["algo"][k] = v if isinstance(v, list) else [v]
    return out


def with_model_dimred(params: dict, method, *, regimes: int = 3) -> dict:
    """Use classical models as soft embeddings into kernel_ruptures."""
    out = copy.deepcopy(params)
    out["algo"]["detection_backend"] = [str(DetectionBackend.KERNEL_RUPTURES)]
    out["algo"]["dimred"] = [str(method)]
    out["algo"]["regimes"] = [regimes]
    return out


def with_tft(
    params: dict,
    *,
    rank: int = 8,
    encoder_length: int = 20,
    prediction_length: int = 5,
    max_epochs: int = 1,
    batch_size: int = 16,
    mode: str = "multivariate",
) -> dict:
    """TFT attention embeddings → kernel_ruptures (smoke-friendly defaults)."""
    out = copy.deepcopy(params)
    out["algo"]["detection_backend"] = [str(DetectionBackend.KERNEL_RUPTURES)]
    out["algo"]["dimred"] = ["tft"]
    out["algo"]["rank"] = [rank]
    out["algo"]["rank_selection_method"] = ["user_specified"]
    out["algo"]["tft_encoder_length"] = [encoder_length]
    out["algo"]["tft_prediction_length"] = [prediction_length]
    out["algo"]["tft_max_epochs"] = [max_epochs]
    out["algo"]["tft_batch_size"] = [batch_size]
    out["algo"]["tft_mode"] = [mode]
    out["algo"]["num_features"] = [30]
    out["algo"]["depth"] = [1]
    return out


def summarize(res, label: str, df: pl.DataFrame) -> None:
    dates = frames.dates_series(df).to_list()
    print(f"[{label}] kept={res.bkpts}  invalid={res.invalid_bkpts}")
    for b in res.bkpts:
        print(f"  bkpt {b} → {dates[b]}")


def run_g1(
    df: pl.DataFrame,
    params: dict,
    label: str,
    plot_vars: list[str],
    *,
    return_fig: bool = False,
):
    """Graph 1 via public single-pass API.

    Returns ``proc`` by default. Pass ``return_fig=True`` to also get the plotnine
    ggplot (without auto-displaying it) for explicit notebook inspection.
    """
    print(
        f"=== Graph 1 · {label} · backend={params['algo'].get('detection_backend')} "
        f"dimred={params['algo']['dimred']} "
        f"detector={params['algo'].get('regime_detection_algorithm')} "
        f"search={params['algo'].get('search_method')} "
        f"test={params['test'].get('choice')} ==="
    )
    proc = run_single_segmentation(df, params)
    summarize(proc, label, df)
    metrics = params.get("metrics", {})
    img_dir = metrics.get("image_dir") or metrics.get("dir")
    if return_fig:
        plot_mode = "write" if img_dir else "display"
        plot_emit = bool(img_dir)
    else:
        plot_mode = "display_and_write" if img_dir else "display"
        plot_emit = True
    fig = plot_regimes(
        df,
        proc,
        variables=plot_vars[:2],
        title=f"Graph 1 · {label}",
        mode=plot_mode,
        img_dir=str(img_dir) if img_dir else None,
        emit=plot_emit,
    )
    if return_fig:
        return proc, fig
    return proc


def run_g2(
    df: pl.DataFrame,
    params: dict,
    seed_res,
    out_dir: Path,
    label: str,
    plot_vars: list[str],
    *,
    max_iter: int = 3,
    threshold: float = 1e-6,
    score_method: str = "mse_to_mean",
    score: dict | None = None,
    return_figs: bool = False,
):
    """Graph 2 via refine_regimes, seeded from a Graph 1 SegmentResults.

    Returns ``out_dir`` by default. Pass ``return_figs=True`` to also get
    ``refined`` and a ``figs`` dict mapping string keys to plotnine ggplots
    (``retrain_iteration_*`` heatmaps + ``regime``).
    """
    out_dir.mkdir(parents=True, exist_ok=True)
    g2 = copy.deepcopy(params)
    g2["metrics"]["dir"] = str(out_dir)
    g2["metrics"]["image_dir"] = str(out_dir)
    g2["metrics"]["mode"] = "display_and_write"
    g2["metrics"]["plot"] = True
    g2["retrain"] = {
        "interactive": False,
        "features": ["__auto__"],
        "num_worst_features": min(5, frames.n_features(df)),
        "threshold": threshold,
        "max_iter": max_iter,
        "score_method": score_method,
        "score": dict(score or {}),
        "regimes_df": None,
    }
    print(f"=== Graph 2 · {label} · score_method={score_method} · seeding from Graph 1 ===")
    print(seed_regimes_from_results(df, seed_res).to_dicts())
    refined = refine_regimes(df, g2, seed=seed_res)
    figs: dict = {}
    if refined is not None:
        summarize(refined, f"{label} Graph 2", df)
        if return_figs:
            figs.update(getattr(refined, "plots", None) or {})
            figs["regime"] = plot_regimes(
                df,
                refined,
                variables=plot_vars[:2],
                title=f"Graph 2 · {label}",
                mode="write",
                img_dir=str(out_dir),
                emit=False,
            )
        else:
            plot_regimes(
                df,
                refined,
                variables=plot_vars[:2],
                title=f"Graph 2 · {label}",
                mode="display",
            )
    if not return_figs:
        pngs = sorted(out_dir.rglob("retrain_iteration_*.png"))[:6]
        print(f"Graph 2 artifacts under {out_dir}")
        for p in pngs:
            print(" ", p.relative_to(out_dir))
            try:
                display(Image(filename=str(p)))
            except Exception as exc:
                print("  (could not display)", exc)
    else:
        print(f"Graph 2 artifacts under {out_dir} ({len(figs)} plotnine figure(s))")
        print(" fig keys:", sorted(figs))
    if return_figs:
        return out_dir, refined, figs
    return out_dir


print("Helpers ready: run_g1/g2, with_dimred/search/test/ess/classical/model_dimred/tft")
print("Enums:", list(DetectionBackend), list(ClassicalDetector)[:4], "...")

In [ ]:
def equity_params(img_dir: Path) -> dict:
    params = load_core_params(img_dir)
    params["metrics"]["features_to_plot"] = plot_cols
    # Returns-based equity panels: shorter min regime than log-price levels (was 20)
    params["algo"]["min_regime_length"] = [20]
    params["algo"]["depth"] = [4]
    params["test"]["significance_level"] = [0.2]
    params["test"]["window"] = [{"method": "user_specified", "window": 60}]
    params["test"]["sample_size"] = [{"method": "user_specified", "num_samples": 60}]
    return params

---
# Part A — PCA (baseline)


## A.1 Graph 1 (PCA)


In [ ]:
params_pca = with_dimred(equity_params(OUT_DIR / "pca"), "pca")
proc_pca, fig_pca = run_g1(features_df, params_pca, "PCA", plot_cols, return_fig=True)
fig_pca

## A.2 Graph 2 (seeded from PCA)

Auto-retrain via a feature×regime **score heatmap** (default `mse_to_mean` / L2).
Part N swaps `retrain.score_method` for panel-friendly alternatives.


In [ ]:
g2_pca_dir, proc_pca_g2, g2_pca_figs = run_g2(
    features_df,
    params_pca,
    proc_pca,
    OUT_DIR / "pca" / "graph2",
    "PCA",
    plot_cols,
    max_iter=3,
    return_figs=True,
)
g2_pca_figs["regime"]


In [ ]:
g2_pca_figs.keys()

---
# Part B — Kernel PCA


## B.1 Graph 1 (kPCA)


In [ ]:
params_kpca = with_dimred(equity_params(OUT_DIR / "kpca"), "kpca")
proc_kpca, g1_kpca = run_g1(features_df, params_kpca, "kPCA", plot_cols, return_fig=True)


In [ ]:
g1_kpca

## B.2 Graph 2 (seeded from kPCA)


In [ ]:
g2_kpca_dir = run_g2(
    features_df, params_kpca, proc_kpca, OUT_DIR / "kpca" / "graph2", "kPCA", plot_cols, max_iter=3
)

---
# Part C — DMD


## C.1 Graph 1 (DMD)


In [ ]:
params_dmd = with_dimred(equity_params(OUT_DIR / "dmd"), "dmd")
proc_dmd, g1_dmd = run_g1(features_df, params_dmd, "DMD", plot_cols, return_fig=True)


In [ ]:
g1_dmd

## C.2 Graph 2 (seeded from DMD)


In [ ]:
g2_dmd_dir, proc_dmd_g2, g2_dmd_figs = run_g2(
    features_df,
    params_dmd,
    proc_dmd,
    OUT_DIR / "dmd" / "graph2",
    "DMD",
    plot_cols,
    max_iter=3,
    return_figs=True,
)
g2_dmd_figs["regime"]

---
# Part D — t-SNE

Nonlinear manifold embedding (`rank=2`) → RFF → PELT → MMD, then Graph 2.


## D.1 Graph 1 (t-SNE)


In [ ]:
params_tsne = with_dimred(equity_params(OUT_DIR / "tsne"), "tsne")
proc_tsne, g1_tsne = run_g1(features_df, params_tsne, "t-SNE", plot_cols, return_fig=True)


In [ ]:
g1_tsne

## D.2 Graph 2 (seeded from t-SNE)


In [ ]:
g2_tsne_dir, proc_tsne_g2, g2_tsne_figs = run_g2(
    features_df, params_tsne, proc_tsne, OUT_DIR / "tsne" / "graph2", "t-SNE", plot_cols, max_iter=3, return_figs=True
)


In [ ]:
g2_tsne_figs['regime']

---
# Part E — UMAP

Optional `umap-learn` dependency. Skips cleanly when unavailable.


## E.1 Graph 1 (UMAP)


In [ ]:
try:
    import umap  # noqa: F401
    _UMAP_OK = True
except Exception as exc:
    _UMAP_OK = False
    print("UMAP stack unavailable — skipping Part E:", exc)

if _UMAP_OK:
    params_umap = with_dimred(equity_params(OUT_DIR / "umap"), "umap")
    proc_umap, g1_umap = run_g1(features_df, params_umap, "UMAP", plot_cols, return_fig=True)
else:
    proc_umap = proc_pca


In [ ]:
g1_umap

## E.2 Graph 2 (seeded from UMAP)


In [ ]:
if _UMAP_OK:
    g2_umap_dir, proc_umap_g2, g2_umap_figs = run_g2(
        features_df, params_umap, proc_umap, OUT_DIR / "umap" / "graph2", "UMAP", plot_cols, max_iter=3,
        return_figs=True
    )
else:
    print("Skipping UMAP Graph 2")


In [ ]:
g2_umap_figs['regime']

---
# Part F — Search methods (Binseg / BottomUp / WBS / BOCPD)

Same PCA + MMD as Part A; only `algo.search_method` changes (incl. **WBS** / **BOCPD**).


## F.1 Binseg


In [ ]:
params_binseg = with_search(equity_params(OUT_DIR / "binseg"), SearchMethod.BINSEG)
params_binseg = with_dimred(params_binseg, "pca")
proc_binseg = run_g1(features_df, params_binseg, "Binseg", plot_cols)


## F.2 BottomUp


In [ ]:
params_bottomup = with_search(equity_params(OUT_DIR / "bottomup"), SearchMethod.BOTTOMUP)
params_bottomup = with_dimred(params_bottomup, "pca")
proc_bottomup = run_g1(features_df, params_bottomup, "BottomUp", plot_cols)


## F.3 Wild Binary Segmentation (WBS)


In [ ]:
params_wbs = with_search(
    equity_params(OUT_DIR / "wbs"),
    SearchMethod.WBS,
    wbs_n_intervals=200,
    random_state=42,
)
params_wbs = with_dimred(params_wbs, "pca")
proc_wbs = run_g1(features_df, params_wbs, "WBS", plot_cols)


## F.4 Bayesian Online Changepoint Detection (BOCPD)


In [ ]:
params_bocpd = with_search(
    equity_params(OUT_DIR / "bocpd"),
    SearchMethod.BOCPD,
    bocpd_hazard=0.01,
    bocpd_threshold=0.4,
    bocpd_max_run=200,
)
params_bocpd = with_dimred(params_bocpd, "pca")
proc_bocpd = run_g1(features_df, params_bocpd, "BOCPD", plot_cols)


---
# Part G — Statistical tests

Swap `test.choice`: energy distance, unbiased / linear-time MMD, Hotelling T²,
multivariate CUSUM, KS on PCA scores.


## G.1 Energy distance


In [ ]:
params_energy = with_test(
    with_dimred(equity_params(OUT_DIR / "energy"), "pca"),
    StatTest.ENERGY_DISTANCE,
)
proc_energy = run_g1(features_df, params_energy, "energy_distance", plot_cols)


## G.2 Unbiased MMD


In [ ]:
params_mmd_u = with_test(
    with_dimred(equity_params(OUT_DIR / "mmd_unbiased"), "pca"),
    StatTest.MMD_UNBIASED,
)
proc_mmd_u = run_g1(features_df, params_mmd_u, "mmd_unbiased", plot_cols)


## G.3 Linear-time MMD


In [ ]:
params_mmd_linear = with_test(equity_params(OUT_DIR / "mmd_linear"), StatTest.MMD_LINEAR)
params_mmd_linear = with_dimred(params_mmd_linear, "pca")
proc_mmd_lin = run_g1(features_df, params_mmd_linear, "mmd_linear", plot_cols)


## G.4 Hotelling T²


In [ ]:
params_hotelling_t2 = with_test(equity_params(OUT_DIR / "hotelling_t2"), StatTest.HOTELLING_T2)
params_hotelling_t2 = with_dimred(params_hotelling_t2, "pca")
proc_hotelling = run_g1(features_df, params_hotelling_t2, "hotelling_t2", plot_cols)


## G.5 Multivariate CUSUM


In [ ]:
params_multivariate_cusum = with_test(equity_params(OUT_DIR / "multivariate_cusum"), StatTest.MULTIVARIATE_CUSUM)
params_multivariate_cusum = with_dimred(params_multivariate_cusum, "pca")
proc_mcusum = run_g1(features_df, params_multivariate_cusum, "multivariate_cusum", plot_cols)


## G.6 KS on PCA scores


In [ ]:
params_ks_pca = with_test(equity_params(OUT_DIR / "ks_pca"), StatTest.KS_PCA)
params_ks_pca = with_dimred(params_ks_pca, "pca")
proc_ks_pca = run_g1(features_df, params_ks_pca, "ks_pca", plot_cols)


---
# Part H — ESS window


In [ ]:
params_ess = with_ess_window(
    with_dimred(equity_params(OUT_DIR / "ess"), "pca"),
    ess_fraction=0.25,
    min_window=20,
    max_window=100,
)
# equity_params sets a fixed window; with_ess_window overwrites it
proc_ess = run_g1(features_df, params_ess, "ESS window", plot_cols)


---
# Part I — Classical hard-label detectors

`detection_backend: classical`. Hard labels → breakpoints;
Graph 2 can still seed from the result.


## I.1 k-means (classical)


In [ ]:
params_ckmeans = with_classical(
    equity_params(OUT_DIR / "classical_kmeans"),
    ClassicalDetector.KMEANS,
    regimes=3,
    random_state=42,
)
proc_ckmeans, proc_ckmeans_fig = run_g1(features_df, params_ckmeans, "classical kmeans", plot_cols, return_fig=True)


## I.2 HMM (classical)


In [ ]:
params_chmm = with_classical(
    equity_params(OUT_DIR / "classical_hmm"),
    ClassicalDetector.HMM,
    regimes=3,
    hmm_emissions="gaussian",
    hmm_n_iter=50,
)
proc_chmm = run_g1(features_df, params_chmm, "classical HMM", plot_cols)


## I.3 Jump model (classical)


In [ ]:
params_cjump = with_classical(
    equity_params(OUT_DIR / "classical_jump_model"),
    ClassicalDetector.JUMP_MODEL,
    regimes=3,
    jump_penalty=5.0,
    jump_max_iter=20,
)
proc_cjump = run_g1(features_df, params_cjump, "classical jump_model", plot_cols)


## I.4 Sticky HDP-HMM (classical)


In [ ]:
params_chdp = with_classical(
    equity_params(OUT_DIR / "classical_sticky_hdp_hmm"),
    ClassicalDetector.STICKY_HDP_HMM,
    regimes=3,
)
proc_chdp = run_g1(features_df, params_chdp, "classical sticky_hdp_hmm", plot_cols)


## I.5 GARCH volatility regimes (classical)


In [ ]:
params_cgarch = with_classical(
    equity_params(OUT_DIR / "classical_garch"),
    ClassicalDetector.GARCH,
    regimes=2,
    garch_p=1,
    garch_q=1,
)
proc_cgarch = run_g1(features_df, params_cgarch, "classical garch", plot_cols)


## I.6 Graph 2 seeded from classical k-means


In [ ]:
g2_ckmeans_dir = run_g2(
    features_df,
    params_ckmeans,
    proc_ckmeans,
    OUT_DIR / "classical_kmeans" / "graph2",
    "classical kmeans",
    plot_cols,
    max_iter=2,
)


---
# Part J — Classical models as soft dimred

`algo.dimred: [kmeans|hmm]` with `detection_backend: kernel_ruptures`.


## J.1 k-means dimred


In [ ]:
params_kmeans_dim = with_model_dimred(
    equity_params(OUT_DIR / "kmeans_dimred"),
    "kmeans",
    regimes=3,
)
proc_kmeans_dim = run_g1(features_df, params_kmeans_dim, "kmeans dimred", plot_cols)


## J.2 HMM dimred


In [ ]:
params_hmm_dim = with_model_dimred(
    equity_params(OUT_DIR / "hmm_dimred"),
    "hmm",
    regimes=3,
)
proc_hmm_dim = run_g1(features_df, params_hmm_dim, "HMM dimred", plot_cols)


---
# Part K — TFT dimred (Temporal Fusion Transformer)

TFT attention embeddings as Graph 1 dimred. Smoke settings (1 epoch). Uses
**multivariate** mode so the multi-stock return panel stays tractable.

Equity tickers with `.` (e.g. `CAP.PA`) are sanitized inside `gulfstream.detectors.tft`.


## K.1 Graph 1 (TFT)


In [ ]:
try:
    import torch  # noqa: F401
    import lightning  # noqa: F401
    import pytorch_forecasting  # noqa: F401
    _TFT_OK = True
except ImportError as exc:
    _TFT_OK = False
    print("TFT stack unavailable — skipping Part K:", exc)

if _TFT_OK:
    params_tft = with_tft(equity_params(OUT_DIR / "tft"))
    print(
        "TFT smoke:",
        f"n={features_df.height}",
        f"enc={params_tft['algo']['tft_encoder_length']}",
        f"epochs={params_tft['algo']['tft_max_epochs']}",
    )
    proc_tft = run_g1(features_df, params_tft, "TFT dimred", plot_cols)
else:
    proc_tft = proc_pca


## K.2 Graph 2 seeded from TFT


In [ ]:
if _TFT_OK:
    g2_tft_dir = run_g2(
        features_df,
        params_tft,
        proc_tft,
        OUT_DIR / "tft" / "graph2",
        "TFT",
        plot_cols,
        max_iter=1,
    )
else:
    print("Skipping TFT Graph 2")


---
# Part L — Curve / ICA dimred

**ICA**, **FPCA**, **Nelson–Siegel**, and **dynamic_factor** embeddings into the
default kernel_ruptures stack. Nelson–Siegel treats feature columns as an ordered
grid (1..p when names are not tenors).


## L.1 ICA


In [ ]:
params_ica = with_dimred(equity_params(OUT_DIR / "ica"), "ica")
proc_ica = run_g1(features_df, params_ica, "ICA dimred", plot_cols)

## L.2 Functional PCA


In [ ]:
params_fpca = with_dimred(equity_params(OUT_DIR / "fpca"), "fpca")
proc_fpca = run_g1(features_df, params_fpca, "FPCA dimred", plot_cols)


## L.3 Nelson–Siegel


In [ ]:
params_ns = with_dimred(equity_params(OUT_DIR / "nelson_siegel"), "nelson_siegel")
proc_ns = run_g1(features_df, params_ns, "Nelson–Siegel dimred", plot_cols)


## L.4 Dynamic factor


In [ ]:
params_dfactor = with_dimred(equity_params(OUT_DIR / "dynamic_factor"), "dynamic_factor")
proc_dfactor = run_g1(features_df, params_dfactor, "dynamic_factor dimred", plot_cols)


---
# Part M — Product features (uncertainty, export, events, streaming, panel)

Dashboard / ops knobs on the same equity feature matrix:

1. **Uncertainty** → `bkpt_ci` shaded as **CI ribbons** (`metrics.plot_ci_ribbons`)
2. **Excel** + **NDJSON events** (`export.excel` / `events`, optional path or dir+filename)
3. **Streaming** expanding Graph 1
4. **Panel joint** consensus (column groups when tickers lack `SOURCE_TENOR` names)


## M.1 Uncertainty bands + CI ribbon overlays


In [ ]:
from gulfstream.metrics import uncertainty as uncertainty_mod

product_dir = OUT_DIR / "product"
product_dir.mkdir(parents=True, exist_ok=True)

params_unc = with_dimred(equity_params(product_dir / "uncertainty"), "pca")
params_unc["metrics"]["plot_ci_ribbons"] = True
params_unc["uncertainty"] = {
    "enabled": True,
    "sources": ["bootstrap"],
    "level": 0.9,
    "match_tolerance": 5,
    "n_bootstrap": 4,
    "bootstrap_block": 25,
    "random_state": 42,
}

proc_unc = run_g1(features_df, params_unc, "PCA + uncertainty", plot_cols)
proc_unc = uncertainty_mod.evaluate_uncertainty(
    features_df,
    {**params_unc, "_pipeline_params": params_unc},
    proc_unc,
)
print("bkpt_ci:", proc_unc.bkpt_ci)
fig_graph_1_pca_ci_ribbons = plot_regimes(
    features_df,
    proc_unc,
    variables=plot_cols[:2],
    title="Graph 1 · PCA + CI ribbons",
    mode="display",
    emit=False,
)
fig_graph_1_pca_ci_ribbons


## M.2 Excel export + NDJSON event stream

Optional `export.excel.path` **or** `dir` + `filename`. Example YAML: `config/graph1/graph1_export_events.yaml`.


In [ ]:
import json

from gulfstream.metrics.writers import export_breakpoint_excel
from gulfstream.ops.events import emit_run_events
import pandas as pd

export_dir = product_dir / "export_events"
export_dir.mkdir(parents=True, exist_ok=True)

params_export = copy.deepcopy(params_unc)
params_export["metrics"]["dir"] = str(export_dir)
params_export["metrics"]["image_dir"] = str(export_dir)
params_export["export"] = {
    "excel": {
        "enabled": True,
        "dir": str(export_dir),
        "filename": "bkpt_export.xlsx",
    }
}
params_export["events"] = {
    "enabled": True,
    "dir": str(export_dir),
    "filename": "events.ndjson",
    "append": False,
}

xlsx_path = export_breakpoint_excel(
    proc_unc,
    params_export,
    dates=frames.dates_series(features_df).to_list(),
)
ndjson_path = emit_run_events(params_export, proc_unc)
print("Excel:", xlsx_path)
print("Events:", ndjson_path)

if ndjson_path:
    for line in Path(ndjson_path).read_text(encoding="utf-8").strip().splitlines():
        evt = json.loads(line)
        print(f"  {evt['event']}")

if xlsx_path:
    display(pd.read_excel(xlsx_path, sheet_name="Breakpoints"))
    display(pd.read_excel(xlsx_path, sheet_name="CI"))


## M.3 Streaming Graph 1 (expanding)


In [ ]:
from gulfstream import detect_regimes_incremental

stream_dir = product_dir / "streaming"
stream_dir.mkdir(parents=True, exist_ok=True)

params_stream = with_dimred(equity_params(stream_dir), "pca")
params_stream["metrics"]["plot"] = False
params_stream["streaming"] = {
    "enabled": True,
    "mode": "expanding",
    "step": 60,
    "min_history": 150,
    "lock_prefix": True,
    "match_tolerance": 5,
}

state = None
proc_stream = None
for step_i in range(3):
    proc_stream, state = detect_regimes_incremental(features_df, params_stream, state)
    print(f"step={step_i} last_t={state.last_t} bkpts={proc_stream.bkpts}")

summarize(proc_stream, "streaming (last step)", features_df)
fig_streaming_graph_1_last_step = plot_regimes(
    features_df,
    proc_stream,
    variables=plot_cols[:2],
    title="Streaming Graph 1 (last step)",
    mode="display",
    emit=False,
)
fig_streaming_graph_1_last_step


## M.4 Panel joint breakpoints

Without `SOURCE_TENOR` column names, the panel path splits features into two column groups and takes a majority consensus.


In [ ]:
from gulfstream import detect_regimes_panel

panel_dir = product_dir / "panel"
panel_dir.mkdir(parents=True, exist_ok=True)

params_panel = with_dimred(equity_params(panel_dir), "pca")
params_panel["metrics"]["plot"] = False
params_panel["panel"] = {
    "enabled": True,
    "groupby": "columns",  # fallback split when no source/tenor prefixes
    "combine": "majority",
    "min_group_frac": 0.5,
    "match_tolerance": 5,
}

proc_panel = detect_regimes_panel(features_df, params_panel)
summarize(proc_panel, "panel majority", features_df)
print("panel_support:", proc_panel.panel_support)
fig_panel_joint_column_majority = plot_regimes(
    features_df,
    proc_panel,
    variables=plot_cols[:2],
    title="Panel joint (column majority)",
    mode="display",
    emit=False,
)
fig_panel_joint_column_majority


---
# Part N — Graph 2 retrain score methods

Graph 2 picks the next slice from a **feature × regime** score matrix
(`retrain.score_method`). Default `mse_to_mean` is the legacy L2 heatmap.
**`threshold` is in the chosen score’s units** — retune when switching methods.

| Method | Idea |
|--------|------|
| `mse_to_mean` | L2 to regime mean (default) |
| `mad_to_median` | Robust L1 twin |
| `mse_on_diff` | MSE on first differences (good for log-prices) |
| `factor_residual` | Within-regime PCA residual |
| `hotelling_within` / `cusum_intensity` | Detection-aligned leftovers |
| `energy_split` / `mmd_split` | Best mid-window two-sample split |


## N.1 Compare score matrices on the PCA Graph 1 seed

No Graph 2 run yet — each scorer reports which (feature, regime) it would refine first.


In [ ]:
import numpy as np

from gulfstream.metrics.regime_scores import known_score_methods, score_feature_regime

score_dir = OUT_DIR / "graph2_scores"
score_dir.mkdir(parents=True, exist_ok=True)

feat_for_scores = [c for c in plot_cols if c in frames.feature_columns(features_df)]
score_df = frames.select_features(features_df, feat_for_scores)
bkpts_seed = list(proc_pca.bkpts)
feat_names = frames.feature_columns(score_df)

split_kwargs = {"n_splits": 5, "min_side": 10, "max_rows": 80}
specs = [
    ("mse_to_mean", {}),
    ("mad_to_median", {}),
    ("mse_on_diff", {"diff_order": 1}),
    ("factor_residual", {"n_components": 1}),
    ("hotelling_within", {}),
    ("cusum_intensity", {}),
    ("energy_split", split_kwargs),
    ("mmd_split", {**split_kwargs, "mmd_estimator": "linear"}),
]

rows = []
for method, kwargs in specs:
    mat = score_feature_regime(score_df, bkpts_seed, method, **kwargs)
    fi, ri = np.unravel_index(int(np.argmax(mat)), mat.shape)
    rows.append(
        {
            "score_method": method,
            "worst_feature": feat_names[fi],
            "worst_regime": int(ri),
            "max_score": float(mat[fi, ri]),
            "n_feat": mat.shape[0],
            "n_regimes": mat.shape[1],
        }
    )
    print(f"{method:18s} → {feat_names[fi]} @ regime {ri}  (max={mat[fi, ri]:.4g})")

print("Available methods:", known_score_methods())
score_pick_table = pl.DataFrame(rows)
score_pick_table


## N.2 Graph 2 with `mse_on_diff` and `factor_residual`

Differenced MSE suits drifting return series; factor residual targets indices that
load on the worst latent factor in each regime.


In [ ]:
params_g2_scores = with_dimred(equity_params(score_dir), "pca")
params_g2_scores["metrics"]["plot"] = False

g2_diff_dir = run_g2(
    features_df,
    params_g2_scores,
    proc_pca,
    score_dir / "mse_on_diff",
    "PCA · mse_on_diff",
    plot_cols,
    max_iter=2,
    threshold=1e-6,
    score_method="mse_on_diff",
    score={"diff_order": 1},
)

g2_factor_dir = run_g2(
    features_df,
    params_g2_scores,
    proc_pca,
    score_dir / "factor_residual",
    "PCA · factor_residual",
    plot_cols,
    max_iter=2,
    threshold=1e-6,
    score_method="factor_residual",
    score={"n_components": 1},
)
print("diff heatmaps:", list(g2_diff_dir.rglob("retrain_iteration_*.png"))[:3])
print("factor heatmaps:", list(g2_factor_dir.rglob("retrain_iteration_*.png"))[:3])


## N.3 Graph 2 with `energy_split` / `mmd_split`

Kernel-aligned mid-window split scorers (smoke-friendly `n_splits` / `max_rows`).


In [ ]:
g2_energy_dir = run_g2(
    features_df,
    params_g2_scores,
    proc_pca,
    score_dir / "energy_split",
    "PCA · energy_split",
    plot_cols,
    max_iter=2,
    threshold=1e-6,
    score_method="energy_split",
    score={"n_splits": 5, "min_side": 10, "max_rows": 80},
)

g2_mmd_dir = run_g2(
    features_df,
    params_g2_scores,
    proc_pca,
    score_dir / "mmd_split",
    "PCA · mmd_split",
    plot_cols,
    max_iter=2,
    threshold=1e-6,
    score_method="mmd_split",
    score={"n_splits": 5, "min_side": 10, "max_rows": 80, "mmd_estimator": "linear"},
)
print("energy:", g2_energy_dir)
print("mmd:", g2_mmd_dir)


---
# Comparison

Covering, **adjusted Rand index**, and breakpoint F1 (tolerance = 10 days) against
the **PCA / PELT / MMD** baseline from Part A. Part M product runs and Part N
score picks are included when those cells were executed.


In [ ]:
dates = frames.dates_series(features_df).to_list()
baseline = proc_pca
n = features_df.height

def row(label: str, res) -> dict:
    f1 = breakpoint_precision_recall_f1(baseline.bkpts, res.bkpts, tolerance=10)
    return {
        "run": label,
        "n_bkpts": len(res.bkpts),
        "bkpts": res.bkpts,
        "dates": [str(dates[b]) for b in res.bkpts],
        "covering_vs_pca": covering_metric(baseline.bkpts, res.bkpts, n),
        "ari_vs_pca": adjusted_rand_index(baseline.bkpts, res.bkpts, n),
        "f1_vs_pca": f1["f1"],
        "precision_vs_pca": f1["precision"],
        "recall_vs_pca": f1["recall"],
    }

rows = [
    row("A pca/pelt/mmd", proc_pca),
    row("B kpca", proc_kpca),
    row("C dmd", proc_dmd),
    row("D tsne", proc_tsne),
    row("E umap", proc_umap),
    row("F binseg", proc_binseg),
    row("F bottomup", proc_bottomup),
    row("F wbs", proc_wbs),
    row("F bocpd", proc_bocpd),
    row("G energy", proc_energy),
    row("G mmd_unbiased", proc_mmd_u),
    row("G mmd_linear", proc_mmd_lin),
    row("G hotelling_t2", proc_hotelling),
    row("G multivariate_cusum", proc_mcusum),
    row("G ks_pca", proc_ks_pca),
    row("H ess window", proc_ess),
    row("I classical kmeans", proc_ckmeans),
    row("I classical hmm", proc_chmm),
    row("I classical jump_model", proc_cjump),
    row("I classical sticky_hdp_hmm", proc_chdp),
    row("I classical garch", proc_cgarch),
    row("J kmeans dimred", proc_kmeans_dim),
    row("J hmm dimred", proc_hmm_dim),
    row("K tft dimred", proc_tft),
    row("L ica", proc_ica),
    row("L fpca", proc_fpca),
    row("L nelson_siegel", proc_ns),
    row("L dynamic_factor", proc_dfactor),
]
if "proc_unc" in globals():
    rows.append(row("M uncertainty", proc_unc))
if "proc_panel" in globals():
    rows.append(row("M panel", proc_panel))

summary = pl.DataFrame(rows)
if "score_pick_table" in globals():
    print("Part N score picks (from N.1):")
    display(score_pick_table)
summary


## What to try next

- Change `START` / `END`, `N_TICKERS_PER_INDEX`, or subset `tickers`.
- Combine knobs (e.g. WBS + `hotelling_t2` + ESS).
- Classical `jump_model` / `sticky_hdp_hmm` / `garch` / `hdbscan` / `optics`.
- Curve dimred: `nelson_siegel`, `fpca`, `dynamic_factor`, or `ica`.
- TFT: raise `tft_max_epochs`, try `tft_mode="univariate"`, or use `config/graph1/graph1_tft_dimred.yaml`.
- Product: `config/graph1/graph1_export_events.yaml`, streaming / panel YAMLs.
- Graph 2 scores: `config/graph2/graph2_score_{diff,factor,energy,mmd}.yaml` — retune `retrain.threshold`.
- Raise Graph 2 `max_iter` or lower `threshold`.
